# Modelo Bayesiano de Presencia-Background para el Copetón
Este notebook implementa la regresión logística Bayesiana según lo documentado en `presence_only_logic.md`, e incluye una simulación de recuperación de parámetros para validar la robustez del modelo.

In [ ]:
import pandas as pd
import numpy as np
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import multivariate_normal

print(f'PyMC version: {pm.__version__}')
print(f'ArviZ version: {az.__version__}')

## 1. Validación del Modelo: Datos Simulados (Parameter Recovery)
Antes de correr el modelo con los datos reales, simulamos un conjunto de datos con características y correlaciones típicas de contaminantes urbanos (ej. correlación positiva entre PM y NO2, y negativa con O3). Definimos coeficientes **conocidos** y verificamos si el modelo en PyMC logra recuperarlos correctamente.

In [ ]:
np.random.seed(42)
n_samples = 3500 # Similar al tamaño del dataset original

# Nombres de las variables simuladas
sim_covariates = ['pm10_ugm3', 'pm25_ugm3', 'no2_ppb', 'o3_ppb', 'co_ppm', 'so2_ugm3']
n_covs = len(sim_covariates)

# Matriz de covarianza simulada para inducir correlación
# PM10 y PM2.5 altamente correlacionados, NO2 y CO correlacionados, O3 correlación negativa con NO2
cov_matrix = np.array([
    [ 1.0,  0.8,  0.3, -0.2,  0.4,  0.2], # pm10
    [ 0.8,  1.0,  0.3, -0.2,  0.4,  0.2], # pm25
    [ 0.3,  0.3,  1.0, -0.6,  0.7,  0.4], # no2
    [-0.2, -0.2, -0.6,  1.0, -0.4, -0.1], # o3
    [ 0.4,  0.4,  0.7, -0.4,  1.0,  0.3], # co
    [ 0.2,  0.2,  0.4, -0.1,  0.3,  1.0]  # so2
])

# Generar predictores X simulados (estandarizados para simplificar)
X_sim = multivariate_normal.rvs(mean=np.zeros(n_covs), cov=cov_matrix, size=n_samples)

# DEFINIMOS LOS BETAS VERDADEROS (Lo que el modelo debe adivinar)
true_beta_0 = -1.5 # Prevalencia base baja
true_betas = np.array([
    -0.8,  # PM10 fuerte negativo
    -0.2,  # PM25 ligero negativo
    -0.5,  # NO2 moderado negativo
     0.3,  # O3 positivo
    -0.1,  # CO neutro/ligero negativo
    -0.4   # SO2 moderado negativo
])

# Predictor lineal (Log-Odds)
eta_sim = true_beta_0 + np.dot(X_sim, true_betas)

# Probabilidad verdadera (Logit inverso)
p_sim = 1 / (1 + np.exp(-eta_sim))

# Generar datos de presencia/fondo reales (Bernoulli)
y_sim = np.random.binomial(1, p_sim)

print(f"Presencias simuladas: {y_sim.sum()}, Fondo simulado: {len(y_sim) - y_sim.sum()}")

In [ ]:
# Modelo en PyMC para los datos simulados
with pm.Model() as sim_model:
    # Priors
    beta = pm.Normal('beta', mu=0, sigma=1, shape=n_covs)
    beta_0 = pm.Normal('beta_0', mu=0, sigma=2)
    
    # Predictor Lineal y Logit
    eta = beta_0 + pm.math.dot(X_sim, beta)
    psi = pm.math.invlogit(eta)
    
    # Likelihood
    obs = pm.Bernoulli('obs', p=psi, observed=y_sim)
    
    # Muestreo corto para la simulación (10000 draws es suficiente para NUTS usualmente)
    sim_trace = pm.sample(draws=10000, tune=1000, chains=4, target_accept=0.9, return_inferencedata=True)

# Gráfico de recuperación de parámetros
fig, ax = plt.subplots(figsize=(10, 6))
az.plot_forest(sim_trace, var_names=['beta'], combined=True, ax=ax)
ax.set_yticklabels(sim_covariates[::-1])

# Dibujar puntos rojos donde están los verdaderos betas
y_positions = np.arange(n_covs)
ax.scatter(true_betas[::-1], y_positions, color='red', marker='D', s=100, label='Verdadero Beta', zorder=3)
ax.axvline(0, color='gray', linestyle='--')
plt.title('Parameter Recovery: Betas Estimados vs Betas Verdaderos (Diamantes Rojos)')
plt.legend()
plt.show()

**Conclusión de la Simulación:** Si los diamantes rojos (los valores verdaderos que introdujimos artificialmente) caen dentro de las barras azules (el Highest Density Interval Bayesiano al 94%), significa que **el modelo matemático y el sampler MCMC son metodológicamente correctos** y pueden distinguir las señales incluso bajo alta correlación entre covariables.

## 2. Carga de Datos Reales
Cargamos los datos preprocesados de presencia y fondo (background) del copetón.

In [ ]:
# Ajustar la ruta si es necesario. En Colab puede ser simplemente 'copeton_presence_only_ready.csv'
import os
DATA_PATH = '../data/processed/copeton_presence_only_ready.csv'

if not os.path.exists(DATA_PATH):
    DATA_PATH = 'copeton_presence_only_ready.csv' # Fallback para Colab local

df = pd.read_csv(DATA_PATH)
df.head()

## 3. Preprocesamiento: Estandarización de Covariables Reales
Estandarizamos los contaminantes para mejorar el muestreo de PyMC.

In [ ]:
covariates = ['pm10_ugm3', 'pm25_ugm3', 'no2_ppb', 'o3_ppb', 'co_ppm', 'so2_ugm3']

# Limpiar NaNs temporalmente
df_clean = df.dropna(subset=covariates + ['y']).copy()

# Guardamos medias y desviaciones
means = df_clean[covariates].mean()
stds = df_clean[covariates].std()

# Estandarizar
X_std = (df_clean[covariates] - means) / stds
y_obs = df_clean['y'].values

print(f'Datos reales listos: {len(df_clean)} registros (Presencias: {int(y_obs.sum())}, Fondo: {len(y_obs) - int(y_obs.sum())})')

## 4. Modelo Bayesiano en PyMC (Datos Reales)

In [ ]:
with pm.Model() as bayesian_pb_model:
    X_data = pm.Data('X', X_std.values)
    beta = pm.Normal('beta', mu=0, sigma=1, shape=X_std.shape[1])
    beta_0 = pm.Normal('beta_0', mu=0, sigma=2)
    eta = beta_0 + pm.math.dot(X_data, beta)
    psi = pm.math.invlogit(eta)
    obs = pm.Bernoulli('obs', p=psi, observed=y_obs)
    
pm.model_to_graphviz(bayesian_pb_model)

## 5. Muestreo MCMC de Alta Resolución
Corremos las cadenas de Markov con alta densidad. **Configuración: 4 cadenas con 10,000 simulaciones (draws) cada una**, tal como lo definimos metodológicamente. 

*Nota Computacional:* En NUTS, 10,000 draws toma un buen tiempo. PyMC usa gradient-based sampling, por lo que 10,000 iteraciones producen un tamaño de muestra efectivo inmenso.

In [ ]:
with bayesian_pb_model:
    # Muestreo de alta densidad (10000 draws x 4 chains = 40,000 muestras posteriores)
    trace = pm.sample(
        draws=10000, 
        tune=1000, 
        chains=4, 
        target_accept=0.9, 
        return_inferencedata=True
    )

## 6. Diagnósticos de Convergencia

In [ ]:
az.plot_trace(trace)
plt.tight_layout()
plt.show()

In [ ]:
az.summary(trace)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
az.plot_forest(trace, var_names=['beta'], combined=True, ax=ax)
ax.set_yticklabels(covariates[::-1])
ax.axvline(0, color='red', linestyle='--')
plt.title('Efectos Estandarizados Reales (Betas)')
plt.show()

## 7. Interpretación Ecológica (Odds Ratios)
Transformamos los betas para interpretarlos en la escala original de los contaminantes.

In [ ]:
summary = az.summary(trace, var_names=['beta'])
beta_std_means = summary['mean'].values
beta_orig_means = beta_std_means / stds.values
odds_ratios = np.exp(beta_orig_means)

or_df = pd.DataFrame({
    'Contaminante': covariates,
    'Beta_Std': beta_std_means,
    'Beta_Original': beta_orig_means,
    'Odds_Ratio': odds_ratios
})

or_df